# PyHealth Data API Tutorial

This notebook covers **`pyhealth.data`** — the foundational layer of PyHealth for representing longitudinal patient records.

You will learn:
- How to create and work with **`Event`** objects representing individual clinical events
- How to build a **`Patient`** object from a structured polars DataFrame
- How to query a patient's event history using **`get_events()`** with powerful filters

---

In [ ]:
!pip install pyhealth

## Overview

In PyHealth, a patient's medical record is modeled as a collection of **events** over time. Each event belongs to a typed category (e.g., `'diagnosis'`, `'lab'`, `'note'`) and carries arbitrary attributes (ICD codes, numeric values, free text, etc.).

The two core classes are:

| Class | Description |
|-------|-------------|
| `Event` | A single timestamped clinical occurrence with typed attributes |
| `Patient` | A patient identified by `patient_id`, holding all events in a polars DataFrame |

These classes are optimized for efficient time-range and attribute filtering, using binary search on sorted timestamps and pre-built event-type partitions.

In [ ]:
from datetime import datetime
import polars as pl
from pyhealth.data import Event, Patient

---
## Part 1: The `Event` Class

An `Event` is a **frozen dataclass** (immutable after creation) with three fields:

```
Event(
    event_type: str,          # category label, e.g. 'diagnosis', 'lab', 'note'
    timestamp:  datetime,     # when the event occurred
    attr_dict:  dict          # arbitrary key-value attributes (set via **kwargs)
)
```

### Creating Events

You can create events for any clinical modality — diagnoses, labs, clinical notes, procedures, medications, etc.

In [ ]:
# --- Diagnosis event (ICD-10-CM code) ---
diagnosis_event = Event(
    event_type="diagnosis",
    timestamp=datetime(2024, 1, 15, 10, 30),
    icd_code="E11.9",
    description="Type 2 diabetes mellitus without complications",
    source="outpatient"
)

# --- Lab event (numeric result) ---
lab_event = Event(
    event_type="lab",
    timestamp=datetime(2024, 1, 15, 11, 0),
    name="HbA1c",
    value=8.5,
    unit="%",
    status="abnormal"
)

# --- Clinical note event (free text) ---
note_event = Event(
    event_type="note",
    timestamp=datetime(2024, 1, 15, 14, 0),
    note_type="Discharge Summary",
    text=(
        "Patient is a 58-year-old male with Type 2 diabetes mellitus. "
        "HbA1c measured at 8.5%, above the target threshold of 7.0%. "
        "Medication regimen adjusted: metformin increased to 1000mg BID."
    )
)

print("Created 3 events:")
print(f"  {diagnosis_event.event_type} at {diagnosis_event.timestamp}")
print(f"  {lab_event.event_type} at {lab_event.timestamp}")
print(f"  {note_event.event_type} at {note_event.timestamp}")

### Accessing Event Attributes

PyHealth supports three access styles — pick whichever is most readable in context:

In [ ]:
# Style 1: subscript (dict-like)
print("[subscript] icd_code:", diagnosis_event["icd_code"])
print("[subscript] timestamp:", diagnosis_event["timestamp"])

# Style 2: dot notation (attribute-like)
print("[dot]       description:", diagnosis_event.description)
print("[dot]       source:",      diagnosis_event.source)

# Style 3: containment check
print("\nAttribute existence:")
print("  'icd_code' in diagnosis_event:", "icd_code" in diagnosis_event)   # True
print("  'severity' in diagnosis_event:", "severity" in diagnosis_event)   # False
print("  'event_type' in diagnosis_event:", "event_type" in diagnosis_event) # always True

In [ ]:
# The attr_dict is the underlying storage for all custom attributes
print("Lab event attr_dict:", lab_event.attr_dict)
print("Lab value:", lab_event.value)
print("Lab unit:",  lab_event.unit)

### Creating an Event from a Dictionary

`Event.from_dict()` is used internally by `Patient.get_events()` when converting polars DataFrame rows back to `Event` objects. You can also use it directly.

The dictionary must follow the column naming convention used by `Patient` (explained in Part 2):

In [ ]:
# Event.from_dict expects the {event_type}/{attr} column naming convention
row_dict = {
    "event_type": "lab",
    "timestamp": datetime(2024, 3, 10, 9, 30),
    "lab/name": "Blood Glucose",
    "lab/value": 320.0,
    "lab/unit": "mg/dL",
    "lab/status": "critical",
    # columns from other event types are simply ignored
    "diagnosis/icd_code": None,
}

glucose_event = Event.from_dict(row_dict)
print("Reconstructed event type:", glucose_event.event_type)
print("Glucose value:", glucose_event.value, glucose_event.unit)
print("Status:", glucose_event.status)

---
## Part 2: The `Patient` Class

A `Patient` holds all events for a single patient in a **polars DataFrame** (`data_source`). Using polars enables fast vectorized filtering and binary search on sorted timestamps.

### Column Naming Convention

The DataFrame must use a specific column structure:

```
Column name format:   {event_type}/{attribute_name}
```

For example:
- `diagnosis/icd_code` — the `icd_code` attribute of a `diagnosis` event
- `lab/value` — the `value` attribute of a `lab` event
- `note/text` — the `text` attribute of a `note` event

Rows belonging to a different event type will have **null** values in columns that don't belong to them. This sparse layout allows a single unified DataFrame to store heterogeneous event types efficiently.

```
 event_type | timestamp   | diagnosis/icd_code | lab/name | lab/value | note/text
 -----------|-------------|--------------------|-----------|-----------|-----------
 diagnosis  | 2024-01-15  | E11.9              | null      | null      | null
 lab        | 2024-01-15  | null               | HbA1c     | 8.5       | null
 note       | 2024-01-15  | null               | null      | null      | Patient...
```

In [ ]:
# Build a DataFrame representing a diabetic patient's record across two visits
data = pl.DataFrame(
    {
        # ── required columns ──────────────────────────────────────────────────
        "event_type": [
            "diagnosis",  "diagnosis",
            "lab",        "lab",        "lab",
            "note",
        ],
        "timestamp": [
            datetime(2024, 1, 15, 10, 30),   # Visit 1 – diagnosis
            datetime(2024, 3, 10,  9,  0),   # Visit 2 – diagnosis
            datetime(2024, 1, 15, 11,  0),   # Visit 1 – HbA1c
            datetime(2024, 3, 10,  9, 30),   # Visit 2 – blood glucose
            datetime(2024, 6,  5, 10,  0),   # Visit 3 – blood glucose
            datetime(2024, 1, 15, 14,  0),   # Visit 1 – discharge note
        ],
        # ── diagnosis columns ────────────────────────────────────────────────
        "diagnosis/icd_code": ["E11.9", "E11.65", None, None, None, None],
        "diagnosis/description": [
            "Type 2 DM without complications",
            "Type 2 DM with hyperglycemia",
            None, None, None, None,
        ],
        # ── lab columns ───────────────────────────────────────────────────────
        "lab/name":   [None, None, "HbA1c",         "Blood Glucose", "Blood Glucose", None],
        "lab/value":  [None, None, 8.5,              320.0,           175.0,          None],
        "lab/unit":   [None, None, "%",              "mg/dL",         "mg/dL",        None],
        "lab/status": [None, None, "abnormal",       "critical",      "high",         None],
        # ── note columns ─────────────────────────────────────────────────────
        "note/note_type": [None, None, None, None, None, "Discharge Summary"],
        "note/text": [
            None, None, None, None, None,
            "58-year-old male with T2DM. HbA1c 8.5%. Metformin increased to 1000mg BID.",
        ],
    }
)

print("DataFrame shape:", data.shape)
print()
data

In [ ]:
# Instantiate the Patient
patient = Patient(patient_id="P001", data_source=data)

print("Patient ID:", patient.patient_id)
print("Total rows in data_source:", len(patient.data_source))
print("Event type partitions:", list(patient.event_type_partitions.keys()))

---
## Part 3: Querying Events with `get_events()`

```python
patient.get_events(
    event_type: Optional[str] = None,        # filter by category
    start:      Optional[datetime] = None,   # inclusive start time
    end:        Optional[datetime] = None,   # inclusive end time
    filters:    Optional[List[tuple]] = None, # attribute filters: [(attr, op, val), ...]
    return_df:  bool = False,                # True → polars DataFrame; False → List[Event]
)
```

**Performance notes:**
- Filtering by `event_type` uses a pre-built partition dict → **O(1)**
- Filtering by time range uses binary search on sorted timestamps → **O(log n)**
- Attribute `filters` are applied afterwards → **O(k)** for k matching rows

In [ ]:
# 3a. Get ALL events (returns a List[Event])
all_events = patient.get_events()
print(f"Total events: {len(all_events)}")
for e in all_events:
    print(f"  [{e.event_type}] {e.timestamp}")

In [ ]:
# 3b. Filter by event_type
diagnoses = patient.get_events(event_type="diagnosis")
print(f"Diagnosis events: {len(diagnoses)}")
for dx in diagnoses:
    print(f"  {dx.timestamp.date()} — {dx['icd_code']}: {dx['description']}")

In [ ]:
# 3c. Filter by time range (inclusive on both ends)
labs_q1 = patient.get_events(
    event_type="lab",
    start=datetime(2024, 1, 1),
    end=datetime(2024, 3, 31),
)
print(f"Lab events in Q1 2024: {len(labs_q1)}")
for lab in labs_q1:
    print(f"  {lab.timestamp.date()} — {lab['name']}: {lab['value']} {lab['unit']} ({lab['status']})")

In [ ]:
# 3d. Attribute filters: only abnormal or critical lab results
# Format: [(attribute_name, operator, value), ...]
# Supported operators: '==', '!=', '<', '<=', '>', '>='
abnormal_labs = patient.get_events(
    event_type="lab",
    filters=[("status", "!=", "high")]  # exclude 'high', keep 'abnormal' and 'critical'
)
print(f"Non-high abnormal labs: {len(abnormal_labs)}")
for lab in abnormal_labs:
    print(f"  {lab['name']}: {lab['value']} {lab['unit']} — {lab['status']}")

In [ ]:
# 3e. Multiple filters (AND logic)
critical_glucose = patient.get_events(
    event_type="lab",
    filters=[
        ("name", "==", "Blood Glucose"),
        ("value", ">", 200.0),
    ]
)
print(f"Critical glucose readings (>200 mg/dL): {len(critical_glucose)}")
for lab in critical_glucose:
    print(f"  {lab.timestamp.date()} — {lab['value']} mg/dL")

In [ ]:
# 3f. Return a polars DataFrame instead of Event objects
lab_df = patient.get_events(event_type="lab", return_df=True)
print("Lab events as DataFrame:")
print(lab_df.select(["timestamp", "lab/name", "lab/value", "lab/unit", "lab/status"]))

---
## Part 4: Realistic Longitudinal Example

Let's build a richer patient record — a 63-year-old with Type 2 diabetes, chronic kidney disease, and polyneuropathy, tracked across three hospital admissions over 18 months.

In [ ]:
# A more complete patient record with three admissions
full_data = pl.DataFrame(
    {
        "event_type": [
            # Admission 1 (Jan 2023)
            "admission",
            "diagnosis", "diagnosis",
            "lab", "lab", "lab",
            "note",
            # Admission 2 (Jul 2023)
            "admission",
            "diagnosis", "diagnosis", "diagnosis",
            "lab", "lab",
            # Admission 3 (Jan 2024)
            "admission",
            "diagnosis",
            "lab",
            "note",
        ],
        "timestamp": [
            # Admission 1
            datetime(2023,  1, 10,  8,  0),
            datetime(2023,  1, 10,  9,  0),
            datetime(2023,  1, 10,  9,  5),
            datetime(2023,  1, 10, 10,  0),
            datetime(2023,  1, 10, 10,  5),
            datetime(2023,  1, 10, 10, 10),
            datetime(2023,  1, 12, 14,  0),
            # Admission 2
            datetime(2023,  7, 20,  8,  0),
            datetime(2023,  7, 20,  9,  0),
            datetime(2023,  7, 20,  9,  5),
            datetime(2023,  7, 20,  9, 10),
            datetime(2023,  7, 20, 10,  0),
            datetime(2023,  7, 20, 10,  5),
            # Admission 3
            datetime(2024,  1,  5,  8,  0),
            datetime(2024,  1,  5,  9,  0),
            datetime(2024,  1,  5, 10,  0),
            datetime(2024,  1,  7, 15,  0),
        ],
        # ── admission columns ──────────────────────────────────────────────────
        "admission/hadm_id": [
            "ADM001", None, None, None, None, None, None,
            "ADM002", None, None, None, None, None,
            "ADM003", None, None, None,
        ],
        "admission/admit_type": [
            "EMERGENCY", None, None, None, None, None, None,
            "ELECTIVE",  None, None, None, None, None,
            "URGENT",    None, None, None,
        ],
        # ── diagnosis columns ──────────────────────────────────────────────────
        "diagnosis/icd_code": [
            None,    "E11.9",  "E11.65",  None, None, None, None,
            None,    "E11.22", "E11.42",  "N18.3", None, None,
            None,    "E11.65", None, None,
        ],
        "diagnosis/description": [
            None,
            "T2DM without complications",
            "T2DM with hyperglycemia",
            None, None, None, None,
            None,
            "T2DM with chronic kidney disease",
            "T2DM with polyneuropathy",
            "Chronic kidney disease, stage 3",
            None, None,
            None,
            "T2DM with hyperglycemia",
            None, None,
        ],
        # ── lab columns ────────────────────────────────────────────────────────
        "lab/name": [
            None, None, None,
            "HbA1c", "eGFR", "Creatinine",
            None,
            None, None, None, None,
            "HbA1c", "eGFR",
            None, None,
            "HbA1c",
            None,
        ],
        "lab/value": [
            None, None, None,
            8.5, 52.0, 1.8,
            None,
            None, None, None, None,
            9.1, 44.0,
            None, None,
            7.8,
            None,
        ],
        "lab/unit": [
            None, None, None,
            "%", "mL/min/1.73m²", "mg/dL",
            None,
            None, None, None, None,
            "%", "mL/min/1.73m²",
            None, None,
            "%",
            None,
        ],
        # ── note columns ───────────────────────────────────────────────────────
        "note/note_type": [
            None, None, None, None, None, None,
            "Discharge Summary",
            None, None, None, None, None, None,
            None, None, None,
            "Progress Note",
        ],
        "note/text": [
            None, None, None, None, None, None,
            "Patient admitted for hyperglycemia management. HbA1c 8.5%. eGFR 52. Discharged on insulin glargine.",
            None, None, None, None, None, None,
            None, None, None,
            "HbA1c improved to 7.8%. eGFR stable. Continue current regimen.",
        ],
    }
)

richPatient = Patient(patient_id="P999", data_source=full_data)
print("Patient P999 record:")
print(f"  Total events: {len(richPatient.get_events())}")
print(f"  Event types present: {list(richPatient.event_type_partitions.keys())}")

In [ ]:
# Query: track HbA1c trend across admissions
hba1c_labs = richPatient.get_events(
    event_type="lab",
    filters=[("name", "==", "HbA1c")]
)
print("HbA1c Trend:")
for lab in hba1c_labs:
    print(f"  {lab.timestamp.date()} — HbA1c: {lab['value']} {lab['unit']}")

In [ ]:
# Query: get all diagnoses within a specific year (2023)
diagnoses_2023 = richPatient.get_events(
    event_type="diagnosis",
    start=datetime(2023, 1, 1),
    end=datetime(2023, 12, 31, 23, 59, 59),
)
print("Diagnoses in 2023:")
for dx in diagnoses_2023:
    print(f"  {dx.timestamp.date()} — {dx['icd_code']}: {dx['description']}")

In [ ]:
# Query: get declining kidney function (eGFR < 50)
low_egfr = richPatient.get_events(
    event_type="lab",
    filters=[
        ("name",  "==", "eGFR"),
        ("value", "<",  50.0),
    ]
)
print("eGFR readings below 50 (worrisome kidney function):")
for lab in low_egfr:
    print(f"  {lab.timestamp.date()} — eGFR: {lab['value']} {lab['unit']}")

---
## Part 5: Bridging to a Real Dataset

| Concept | Key API |
|---------|----------|
| Create an event | `Event(event_type, timestamp, **kwargs)` |
| Access event attribute | `event["key"]`, `event.key`, `"key" in event` |
| Build from raw dict | `Event.from_dict(dict)` |
| Create a patient | `Patient(patient_id, data_source=pl.DataFrame(...))` |
| Get all events | `patient.get_events()` |
| Filter by type | `patient.get_events(event_type="diagnoses_icd")` |
| Filter by time | `patient.get_events(start=..., end=...)` |
| Filter by attribute | `patient.get_events(event_type="prescriptions", filters=[("route", "==", "IV")])` |
| Return as DataFrame | `patient.get_events(return_df=True)` |
| Load from MIMIC-III | `MIMIC3Dataset(root=..., tables=["diagnoses_icd", ...])` |

### Table name = event type

When using a dataset loader like `MIMIC3Dataset`, the `event_type` on every event equals the **table name** from `mimic3.yaml` — not a generic category like `"diagnosis"`. The available attributes on each event are exactly the columns listed under that table's `attributes` key in the YAML.

```
Table name          → event_type           → example attributes
─────────────────────────────────────────────────────────────────
diagnoses_icd       → "diagnoses_icd"      → icd9_code, hadm_id, seq_num
prescriptions       → "prescriptions"      → drug, ndc, dose_val_rx, route
noteevents          → "noteevents"         → text, category, description
admissions          → "admissions"         → hadm_id, admission_type, hospital_expire_flag
icustays            → "icustays"           → icustay_id, first_careunit, outtime
```

When writing a custom task, always check the relevant YAML config to know the exact attribute names available on events from each table.

In [ ]:
from pyhealth.datasets import MIMIC3Dataset

root = "https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III"

dataset = MIMIC3Dataset(
    root=root,
    dataset_name="mimic3",
    tables=[
        "diagnoses_icd",   # ICD-9 diagnosis codes per admission
        "prescriptions",   # medication orders
        "noteevents",      # clinical notes
    ],
)

print(f"Loaded {len(dataset.patients)} patients")

# Grab one patient to explore
patient_id = list(dataset.patients.keys())[0]
patient = dataset.patients[patient_id]
print(f"\nPatient ID: {patient.patient_id}")
print(f"Event types present: {list(patient.event_type_partitions.keys())}")
print(f"Total events:        {len(patient.get_events())}")

In [ ]:
# Pull diagnosis events — event_type matches the table name exactly: "diagnoses_icd"
diagnoses = patient.get_events("diagnoses_icd")
print(f"Diagnosis events: {len(diagnoses)}")
print()

# Each event's attributes come from the 'attributes' list in mimic3.yaml
#   hadm_id, icd9_code, seq_num
for dx in diagnoses[:5]:
    print(f"  [{dx.timestamp.date()}] hadm={dx['hadm_id']}  "
          f"ICD-9={dx['icd9_code']}  seq={dx['seq_num']}")

In [ ]:
# Pull prescription events and filter to a specific route (e.g. IV medications)
prescriptions = patient.get_events("prescriptions")
print(f"Total prescription events: {len(prescriptions)}")

# Attribute filters work the same way on real data
iv_meds = patient.get_events(
    event_type="prescriptions",
    filters=[("route", "==", "IV")],
)
print(f"IV medications: {len(iv_meds)}")
for rx in iv_meds[:5]:
    print(f"  [{rx.timestamp.date()}] {rx['drug']}  dose={rx['dose_val_rx']}  ndc={rx['ndc']}")

In [ ]:
# Clinical note events — attribute 'text' holds the full note body
# Attributes available: text, category, description, hadm_id, storetime (from mimic3.yaml)
notes = patient.get_events("noteevents")
print(f"Note events: {len(notes)}")

discharge_notes = patient.get_events(
    event_type="noteevents",
    filters=[("category", "==", "Discharge summary")],
)
print(f"Discharge summaries: {len(discharge_notes)}")
if discharge_notes:
    first_note = discharge_notes[0]
    print(f"\n  Date: {first_note.timestamp.date()}")
    print(f"  Category: {first_note['category']}")
    print(f"  Text preview: {str(first_note['text'])[:200]}...")

---
## Summary

| Concept | Key API |
|---------|----------|
| Create an event | `Event(event_type, timestamp, **kwargs)` |
| Access event attribute | `event["key"]`, `event.key`, `"key" in event` |
| Build from raw dict | `Event.from_dict(dict)` |
| Create a patient | `Patient(patient_id, data_source=pl.DataFrame(...))` |
| Get all events | `patient.get_events()` |
| Filter by type | `patient.get_events(event_type="lab")` |
| Filter by time | `patient.get_events(start=..., end=...)` |
| Filter by attribute | `patient.get_events(event_type="lab", filters=[("value", ">", 7.0)])` |
| Return as DataFrame | `patient.get_events(return_df=True)` |

In practice you won't create `Patient` objects manually — `MIMIC3Dataset` and other dataset loaders build them from raw EHR tables and expose them through the `set_task()` pipeline. But understanding the underlying API helps when writing custom tasks or debugging.